# 1. Importing libraries

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import make_union
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.base import clone
from sklearn.metrics import balanced_accuracy_score, classification_report

In [ ]:
DATA = 'data/'

# 2. Model training

## 2.1 Vectorize

### 2.1a TF-IDF vectorizer

In [ ]:
tfidf_vectorizer = make_union(
    TfidfVectorizer(analyzer='char', ngram_range=(2, 5), sublinear_tf=True,
                    min_df=2, max_features=300_000, lowercase=False),
    TfidfVectorizer(analyzer='word', ngram_range=(1, 2), sublinear_tf=True,
                    min_df=2, lowercase=False, token_pattern=r"(?u)\b\w+\b|[^\w\s]"),
)

### 2.1b Count vectorizer

In [ ]:
count_vectorizer = make_union(
    CountVectorizer(analyzer='char', ngram_range=(2, 5),
                    min_df=2, max_features=300_000, lowercase=False),
    CountVectorizer(analyzer='word', ngram_range=(1, 2),
                    min_df=2, lowercase=False, token_pattern=r"(?u)\b\w+\b|[^\w\s]"),
)

## 2.2 Classifier models

In [ ]:
def train_model(model, vectorizer, X_train, y_train, X_val):
    X_train_vec = vectorizer.fit_transform(X_train)
    X_val_vec = vectorizer.transform(X_val)

    model.fit(X_train_vec, y_train)
    
    y_pred = model.predict(X_val_vec)
    
    return y_pred

### 2.2a SVM

In [ ]:
svm = LinearSVC(C=0.5, class_weight='balanced')

### 2.2b Logistic regression  

In [ ]:
log_reg = LogisticRegression(C=10, class_weight='balanced', max_iter=2000)

## 2.3 Training the model

In [ ]:
df = pd.read_csv(f'{DATA}/train.csv')

vectorizers = {
    'TF-IDF': tfidf_vectorizer,
    'Count': count_vectorizer,
}

classifiers = {
    'SVM': svm, 
    'Logistic Regression': log_reg,
}

results = {}
all_oofs = {} 

for vec_name, vectorizer in vectorizers.items():
    for model_name, model in classifiers.items():
        oof = pd.Series(index=df.index, dtype=object)
        for train_idx, val_idx in GroupKFold(n_splits=5).split(df, groups=df["prompt_id"]):
            train_df = df.iloc[train_idx]
            val_df = df.iloc[val_idx]

            X_train, y_train = train_df['text'], train_df['model']
            X_val, y_val = val_df['text'], val_df['model']

            oof.iloc[val_idx] = train_model(
                clone(model), 
                clone(vectorizer),
                X_train, y_train, X_val
            )

        score = balanced_accuracy_score(df['model'], oof)
        results[(vec_name, model_name)] = score
        all_oofs[(vec_name, model_name)] = oof
        
        print(f'Vectorizer = {vec_name}, Model = {model_name}, Balanced accuracy = {score:.4f}')
        print(classification_report(df['model'], oof, digits=2))

Vectorizer = TF-IDF, Model = SVM, Balanced accuracy = 0.3382
                        precision    recall  f1-score   support

        claude-fable-5       0.13      0.08      0.10       149
       claude-opus-4-6       0.46      0.42      0.44       150
       claude-opus-4-8       0.24      0.14      0.18       150
         claude-opus-5       0.35      0.67      0.46       150
       claude-sonnet-5       0.33      0.35      0.34       150
gemini-3.1-pro-preview       0.57      0.74      0.64       150
      gemini-3.6-flash       0.42      0.43      0.42       150
      gemini-3.7-flash       0.37      0.43      0.40       150
      gemini-3.8-flash       0.33      0.31      0.32       150
               glm-5.3       0.25      0.17      0.20       150
         glm-5.3-flash       0.24      0.16      0.19       150
               gpt-5.5       0.28      0.28      0.28       150
          gpt-5.6-luna       0.23      0.28      0.25       150
           gpt-5.6-sol       0.33      0.2

In [ ]:
oof = all_oofs[('TF-IDF', 'SVM')]   # change to any combination

conf = pd.crosstab(df["model"], oof, normalize='index').round(2)
fig = px.imshow(conf, text_auto=True, aspect='auto', color_continuous_scale='Blues',
                title='Which models get confused (row = true, column = predicted)',
                labels={'x': 'predicted', 'y': 'true', 'color': 'share'})
fig.update_layout(height=750)
fig.show()

how to read the heatmap: 
- Read the diagonal from top-left to bottom-right, where the row and column are the same model. These are the correct predictions, so each diagonal number is that model’s recall. Ideally the diagonal is dark and everything else is pale.
    - gemini-3.1-pro-preview → gemini-3.1-pro-preview = 0.74: 74% correct, the best of all.
    - claude-fable-5 → claude-fable-5 = 0.08: only 8% correct, the worst.

- Read across a row to see where a model’s mistakes go. Take the glm-5.3 row:
    - 0.17 under glm-5.3: correct
    - 0.25 under claude-opus-5: its biggest share is actually a wrong guess
    - 0.14 under glm-5.3-flash: confused with its twin

    So the classifier calls glm-5.3 “Claude Opus 5” more often than it gets it right.

- Read down a column to see which models get dumped into one label. The claude-opus-5 column has noticeable values from claude-fable-5 (0.21), claude-opus-4-8 (0.15), glm-5.3 (0.25) and glm-5.3-flash (0.19). That’s what makes Opus 5 a catch-all: high recall, but low precision.

- Look for dark blocks off the diagonal. These show groups of models that blur together:
    - The Gemini block: the four Gemini rows and columns are blue among themselves and nearly white everywhere else. The family is easy; telling the flash models apart is the hard part.
    - The GPT-5.6 block: Luna, Sol, Terra and Astra spread their predictions across each other.
    - The Claude + GLM block: the seven rows mix together, which shows the classifier treats them as one family.